In [ ]:
!pip install youtube-transcript-api

In [ ]:
import requests
import urllib3
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [41]:
import requests #handlin internal ssl request
import urllib3
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

video_id = "1bUy-1hGZpI" # only the ID, not full URL
try:
    # If you don't care which language, this returns the best English transcript.
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list) #loops through all subtitle chunks
    print(transcript_list)

#basic code block for ssl bad request
except requests.exceptions.RequestException:
    # Use this fallback when local proxy or certificate settings block YouTube.
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    session = requests.Session()
    session.verify = False
    session.trust_env = False

    #retrying transcript
    transcript_list = YouTubeTranscriptApi(http_client=session).fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript_list)

except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable) as error:
    print(f"No captions available for this video: {error}")

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="now stop me if you've heard this one", start=0.08, duration=3.68), FetchedTranscriptSnippet(text='before but there are a lot of large', start=1.599, duration=4.401), FetchedTranscriptSnippet(text='language models available today and they', start=3.76, duration=4.04), FetchedTranscriptSnippet(text='have their own capabilities and', start=6.0, duration=4.599), FetchedTranscriptSnippet(text='specialities what if I prefer to use one', start=7.8, duration=5.16), FetchedTranscriptSnippet(text='llm to interpret some user queries in my', start=10.599, duration=4.801), FetchedTranscriptSnippet(text='business application but a whole other', start=12.96, duration=5.12), FetchedTranscriptSnippet(text='llm to author a response to those', start=15.4, duration=5.68), FetchedTranscriptSnippet(text='queries well that scenario is exactly', start=18.08, duration=6.68), FetchedTranscriptSnippet(text='what Lang chain caters to Lang chain is', start

In [16]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="now stop me if you've heard this one", start=0.08, duration=3.68), FetchedTranscriptSnippet(text='before but there are a lot of large', start=1.599, duration=4.401), FetchedTranscriptSnippet(text='language models available today and they', start=3.76, duration=4.04), FetchedTranscriptSnippet(text='have their own capabilities and', start=6.0, duration=4.599), FetchedTranscriptSnippet(text='specialities what if I prefer to use one', start=7.8, duration=5.16), FetchedTranscriptSnippet(text='llm to interpret some user queries in my', start=10.599, duration=4.801), FetchedTranscriptSnippet(text='business application but a whole other', start=12.96, duration=5.12), FetchedTranscriptSnippet(text='llm to author a response to those', start=15.4, duration=5.68), FetchedTranscriptSnippet(text='queries well that scenario is exactly', start=18.08, duration=6.68), FetchedTranscriptSnippet(text='what Lang chain caters to Lang chain is', start

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [18]:
len(chunks)

9

In [19]:
chunks[0]

Document(metadata={}, page_content="now stop me if you've heard this one before but there are a lot of large language models available today and they have their own capabilities and specialities what if I prefer to use one llm to interpret some user queries in my business application but a whole other llm to author a response to those queries well that scenario is exactly what Lang chain caters to Lang chain is an open-source orchestration framework for the development of applications that use large language models and it comes in both Python and JavaScript libraries it's it's essentially a generic interface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows now when it was launched by Harrison Chase in October 2022 Lang chain enjoyed a meteoric rise and by June of the following year it was the single fastest growing open- source project on GitHu

In [20]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(chunks, embeddings)

In [21]:
vector_store.index_to_docstore_id

{0: '2052de5d-12d6-44e8-b2f1-76d90c855943',
 1: '24c3e83d-b37b-4867-952d-5bbd7fee9fa8',
 2: 'd4fc7fd1-8752-43af-a654-da4bc52d1b97',
 3: '0043a178-b723-4f95-a864-8f3c8256529b',
 4: '887cf898-4fab-4a6d-8a13-57e50d47786a',
 5: '0005bdbc-ec51-4805-aaa6-8670b31847ff',
 6: '8dacace1-04b3-4eb1-8586-f9ac494c9b23',
 7: 'f086c69a-78b9-4bf8-8ca1-c40ea87248a6',
 8: 'f6bb0a67-159a-4f0c-8b58-d59a69222c76'}

In [22]:
vector_store.get_by_ids(['2052de5d-12d6-44e8-b2f1-76d90c855943'])

[Document(id='2052de5d-12d6-44e8-b2f1-76d90c855943', metadata={}, page_content="now stop me if you've heard this one before but there are a lot of large language models available today and they have their own capabilities and specialities what if I prefer to use one llm to interpret some user queries in my business application but a whole other llm to author a response to those queries well that scenario is exactly what Lang chain caters to Lang chain is an open-source orchestration framework for the development of applications that use large language models and it comes in both Python and JavaScript libraries it's it's essentially a generic interface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows now when it was launched by Harrison Chase in October 2022 Lang chain enjoyed a meteoric rise and by June of the following year it was the single f

In [23]:
#Retriveal
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [24]:
retriever.invoke('What is langchain')

[Document(id='2052de5d-12d6-44e8-b2f1-76d90c855943', metadata={}, page_content="now stop me if you've heard this one before but there are a lot of large language models available today and they have their own capabilities and specialities what if I prefer to use one llm to interpret some user queries in my business application but a whole other llm to author a response to those queries well that scenario is exactly what Lang chain caters to Lang chain is an open-source orchestration framework for the development of applications that use large language models and it comes in both Python and JavaScript libraries it's it's essentially a generic interface for nearly any llm so you have a centralized development environment to build your large language model applications and then integrate them with stuff like data sources and software workflows now when it was launched by Harrison Chase in October 2022 Lang chain enjoyed a meteoric rise and by June of the following year it was the single f

In [25]:
llm = ChatOllama(model="llama3", temperature=0.2)

In [26]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [27]:
question          = "Are application of Langchain are discussed in video? If yes what are they?"
retrieved_docs    = retriever.invoke(question)

In [28]:
retrieved_docs

[Document(id='f6bb0a67-159a-4f0c-8b58-d59a69222c76', metadata={}, page_content="modules can use an llm to autonomously determine the next steps and then take the action that it needs to complete that step using something called RPA or robotic process automation Lang chain is open source and free to use there are also related Frameworks like Lang serve for creating chains as rest apis and Lang Smith which provides tools to monitor evaluate and debug applications essentially Lang Chain's tools and apis simplify the process of building applications that make use of large language models if you have any questions please drop us a line below and if you want to see more videos like this in the future please like And subscribe thanks for watching"),
 Document(id='0043a178-b723-4f95-a864-8f3c8256529b', metadata={}, page_content="a set of examples to guide its responses that's called f shot prompting or it could specify an output format now chains as the name implies are the core of Lang chain 

In [29]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"modules can use an llm to autonomously determine the next steps and then take the action that it needs to complete that step using something called RPA or robotic process automation Lang chain is open source and free to use there are also related Frameworks like Lang serve for creating chains as rest apis and Lang Smith which provides tools to monitor evaluate and debug applications essentially Lang Chain's tools and apis simplify the process of building applications that make use of large language models if you have any questions please drop us a line below and if you want to see more videos like this in the future please like And subscribe thanks for watching\n\na set of examples to guide its responses that's called f shot prompting or it could specify an output format now chains as the name implies are the core of Lang chain workflows they combine llms with other components creating applications by executing a sequence of functions so let's say our application that needs to first o

In [30]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [31]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      modules can use an llm to autonomously determine the next steps and then take the action that it needs to complete that step using something called RPA or robotic process automation Lang chain is open source and free to use there are also related Frameworks like Lang serve for creating chains as rest apis and Lang Smith which provides tools to monitor evaluate and debug applications essentially Lang Chain's tools and apis simplify the process of building applications that make use of large language models if you have any questions please drop us a line below and if you want to see more videos like this in the future please like And subscribe thanks for watching\n\na set of examples to guide its responses that's called f shot prompting or it could specify an output format now chains as the name impl

In [32]:
#Generation
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, applications of LangChain are discussed in the video. According to the transcript, one example of a LangChain workflow is an application that:

"needs to first of all retrieve data from a website then it needs to summarize the text it gets back and then finally it needs to use that summary to answer User submitted questions"

This sequential chain combines LLMS with other components to create an application by executing a sequence of functions.


In [33]:
#CHAIN
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [34]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [35]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [36]:
parallel_chain.invoke('what is langchain api')

{'context': "modules can use an llm to autonomously determine the next steps and then take the action that it needs to complete that step using something called RPA or robotic process automation Lang chain is open source and free to use there are also related Frameworks like Lang serve for creating chains as rest apis and Lang Smith which provides tools to monitor evaluate and debug applications essentially Lang Chain's tools and apis simplify the process of building applications that make use of large language models if you have any questions please drop us a line below and if you want to see more videos like this in the future please like And subscribe thanks for watching\n\nnow stop me if you've heard this one before but there are a lot of large language models available today and they have their own capabilities and specialities what if I prefer to use one llm to interpret some user queries in my business application but a whole other llm to author a response to those queries well 

In [37]:
parser = StrOutputParser()

In [38]:
main_chain = parallel_chain | prompt | llm | parser

In [39]:
main_chain.invoke('Can you summarize the video')

"Here is a summary of the video:\n\nThe video discusses Lang Chain, a framework that streamlines the programming of Large Language Model (LLM) applications. It explains how Lang Chain combines LLMS with other components to create applications by executing a sequence of functions. The video highlights three main components: vectors for storing and retrieving information, text splitters for breaking down text into meaningful chunks, and agents for using language models as reasoning engines.\n\nIt also mentions indexes, which refer to external data sources that can be accessed by LLMS. Additionally, the video touches on Lang Chain's open-source nature and its ability to simplify the process of building applications that utilize large language models.\n\nThe video concludes by highlighting Lang Chain's abstractions, which represent common steps and concepts necessary for working with language models, allowing developers to create complex NLP tasks with minimal code."